In [1]:
print("hello world")

hello world


In [2]:
import yaml
from dotenv import load_dotenv
import os

import warnings
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

# Load config.yaml
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f) 
    #safe_load is used to load the YAML file safely, preventing the execution of arbitrary code.
    
print("✅ Environment loaded successfully")
print("✅ LLM Provider:", config["llm"]["provider"])
print("✅ OpenAI Key exists:", "OPENAI_API_KEY" in os.environ)
print("✅ Google Key exists:", "GOOGLE_API_KEY" in os.environ)

✅ Environment loaded successfully
✅ LLM Provider: openai
✅ OpenAI Key exists: True
✅ Google Key exists: True


In [ ]:
# Test-2: Loader

from utils.loader import load_and_chunk_docs

chunks = load_and_chunk_docs("./data/raw/insurance_docs", chunk_size=80, chunk_overlap=20)
print(f"First chunk sample:\n{chunks[0].page_content[:150]}...") #read only first 150 characters of the first chunk
print(f"Total chunks created: {len(chunks)}")

✅ Loaded 2 docs → 65 chunks
First chunk sample:
Claims Reporting and Processing Procedure

I. Initiate Your Claim...
Total chunks created: 65


In [4]:
# Test-3: Retriever
 
from utils.loader import load_and_chunk_docs
from utils.retriever import create_retriever, load_retriever, load_config

config = load_config("config.yaml")

chunks = load_and_chunk_docs("./data/raw/insurance_docs")

# create or load provider-specific FAISS
retriever = create_retriever(chunks, provider=config["llm"]["provider"], config=config)

# load again to verify
retriever = load_retriever(provider=config["llm"]["provider"], config=config)

✅ Loaded 2 docs → 8 chunks
✅ Created new FAISS index (openai) at: ./data/embeddings/faiss_openai
✅ Loaded FAISS index (openai) from: ./data/embeddings/faiss_openai


In [8]:
# Test-4: Query Re-writer

from utils.query_rewriter import rewrite_query
import yaml

# Confirm config is read correctly
with open("config.yaml") as f:
    config = yaml.safe_load(f)["llm"]
    print("✅ Loaded config:", config)

query = "cashless policy?"

rewritten = rewrite_query(query)
print("\n🔁 Rewritten Query:\n", rewritten)

✅ Loaded config: {'provider': 'openai', 'model_openai': 'gpt-4o-mini', 'model_gemini': 'gemini-2.5-flash', 'temperature': 0.3, 'max_tokens': 1000}

🔁 Rewritten Query:
 What are the key features and implications of a cashless policy, and how does it affect consumers and businesses in today's economy?


In [9]:
# Test-5: hyde-generator

from utils.hyde_generator import generate_hyde_embedding

query = "cashless policy?"
embedding = generate_hyde_embedding(query)

print("\n✅ Embedding shape:", embedding.shape)
print("✅ Example values:", embedding[:5])

Query: cashless policy?

🧪 Synthetic HyDE Answer:  A cashless policy refers to a financial strategy implemented by governments or organizations aimed at reducing or eliminating the use of physical cash in transactions. This policy encourages the adoption of digital payment methods, such as credit and debit cards, mobile wallets, and online banking. The primary goals of a cashless policy include increasing transaction efficiency, enhancing tax compliance, reducing the costs associated with cash handling, and improving security by minimizing the risks of theft and fraud. Countries like Sweden and India have made significant strides in implementing cashless initiatives, often supported by technological advancements and regulatory frameworks.

✅ Embedding shape: (1536,)
✅ Example values: [ 0.02763367 -0.0249176   0.02711487  0.02005005  0.0259552 ]
